# Notebook 05 — Differential Abundance Analysis

Compares microbial genera between:
- **Cancer vs Healthy** for Colorectal and Breast (Prostate has no healthy controls)
- **Cross-cancer** (Colorectal vs Breast vs Prostate cancer samples)

Produces Figures 5–8b and differential abundance result tables.

In [ ]:
# Cell 1 — Imports
import pandas as pd          # pandas: tables and data manipulation (like Excel in Python)
import numpy as np            # numpy: fast math on arrays of numbers
import matplotlib             # matplotlib: the core Python charting library
matplotlib.use('Agg')         # 'Agg' = save plots to files instead of opening windows
import matplotlib.pyplot as plt  # pyplot: easier interface for drawing charts
import seaborn as sns         # seaborn: prettier statistical plots built on matplotlib
from scipy import stats       # scipy.stats: ready-made statistical tests
from scipy.stats import mannwhitneyu  # Mann-Whitney U: compares two groups without assuming normal distribution
import warnings               # warnings: suppresses minor alerts
warnings.filterwarnings('ignore')  # keep output clean — hide harmless warnings
import os                     # os: file and folder path utilities
from itertools import combinations  # combinations: generates all pairs from a list, e.g. (A,B), (A,C), (B,C)

print('Imports complete.')

In [ ]:
# Cell 2 — Paths and load data
BASE_DIR = os.path.join(
    'c:\\', 'MyProjects', 'Project-Proposal', 'PrivateCoach',
    'In Progress', 'Project-BioInformatics', 'Projects',
    'cancer-microbiome', 'Final_Solution'
)
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')   # folder containing processed CSV outputs
FIGURES_DIR = os.path.join(BASE_DIR, 'Figures')   # folder where chart images will be saved

os.makedirs(RESULTS_DIR, exist_ok=True)  # create folders if they don't already exist
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load the genus-level relative abundance table produced by Notebook 02
# Rows = patients/samples, Columns = bacterial genera, Values = fraction 0–1
abund = pd.read_csv(os.path.join(RESULTS_DIR, 'abund_combined_genus.csv'), index_col=0)

# Load the metadata: cancer_type (Colorectal/Breast/Prostate) and condition (Cancer/Healthy)
meta  = pd.read_csv(os.path.join(RESULTS_DIR, 'meta_combined.csv'),         index_col=0)

# Keep only samples that appear in BOTH tables (inner join by sample ID)
common_idx = abund.index.intersection(meta.index)
abund = abund.loc[common_idx]  # filter abundance rows to shared samples
meta  = meta.loc[common_idx]   # filter metadata rows to shared samples

print(f'Abundance matrix : {abund.shape}  (samples x genera)')
print(f'Metadata         : {meta.shape}')
print(f'\nCancer types present: {meta["cancer_type"].unique()}')
print(f'\nSample breakdown:')
print(meta.groupby(['cancer_type', 'condition']).size())  # count samples per cancer type × condition

In [ ]:
# Cell 3 — Mann-Whitney U: Cancer vs Healthy (Colorectal and Breast only)
# Goal: for each bacterium, ask "is its abundance significantly different in cancer vs healthy people?"
# Prostate has no healthy controls in this dataset, so we can only compare Colorectal and Breast

from statsmodels.stats.multitest import multipletests  # for BH FDR correction (reduces false positives)

def run_diff_abundance(abund_df, meta_df, cancer_type_label):
    """Run Mann-Whitney U (Cancer vs Healthy) for a single cancer type.
    Returns a table of all genera with their U statistics, p-values, and fold changes.
    """
    # Get the sample IDs that belong to this cancer type
    ct_mask     = meta_df['cancer_type'] == cancer_type_label
    cancer_idx  = meta_df.index[ct_mask & (meta_df['condition'] == 'Cancer')]   # cancer patient IDs
    healthy_idx = meta_df.index[ct_mask & (meta_df['condition'] == 'Healthy')]  # healthy control IDs

    # Skip this cancer type if one group is missing (e.g., Prostate has no healthy controls)
    if len(cancer_idx) == 0 or len(healthy_idx) == 0:
        print(f'  [SKIP] {cancer_type_label}: cancer={len(cancer_idx)}, healthy={len(healthy_idx)}')
        return pd.DataFrame()

    cancer_data  = abund_df.loc[cancer_idx]   # abundance rows for cancer patients
    healthy_data = abund_df.loc[healthy_idx]  # abundance rows for healthy controls

    results = []
    for genus in abund_df.columns:  # loop over every bacterium column
        c_vals = cancer_data[genus].values   # array of abundances in cancer samples
        h_vals = healthy_data[genus].values  # array of abundances in healthy samples

        # If both groups have zero variance (everyone has the same value), skip the test
        if c_vals.std() == 0 and h_vals.std() == 0:
            u_stat, p_val = np.nan, 1.0  # p=1.0 means "no evidence of difference"
        else:
            try:
                # Mann-Whitney U test: ranks all values together and checks if one group's ranks are higher
                # alternative='greater' tests if cancer abundance > healthy abundance
                u_stat, p_val = mannwhitneyu(c_vals, h_vals, alternative='greater')
            except Exception:
                u_stat, p_val = np.nan, 1.0  # fall back if test fails

        med_c = np.median(c_vals)   # median abundance in cancer group
        med_h = np.median(h_vals)   # median abundance in healthy group

        # Fold change = how many times more (or less) abundant in cancer vs healthy
        # +1e-10 pseudocount prevents division by zero when both medians are 0
        fold_change = (med_c + 1e-10) / (med_h + 1e-10)
        log2_fc     = np.log2(fold_change)  # log2 scale: +1 = 2× more, -1 = 2× less

        results.append({
            'genus':            genus,
            'U_stat':           u_stat,
            'p_value':          p_val,
            'log2_fold_change': log2_fc,
            'median_cancer':    med_c,
            'median_healthy':   med_h
        })

    df = pd.DataFrame(results)
    df = df.dropna(subset=['p_value'])  # remove rows where test failed

    # Benjamini-Hochberg (BH) FDR correction: adjusts p-values to account for testing 259 genera at once
    # Without correction, 5% of genera (~13) would appear significant by random chance alone
    reject, adj_p, _, _ = multipletests(df['p_value'].values, method='fdr_bh')
    df['adjusted_p'] = adj_p  # use adjusted p-values for significance decisions

    # Label direction: enriched = more abundant in cancer, depleted = less abundant in cancer
    df['direction'] = np.where(df['log2_fold_change'] > 0, 'enriched', 'depleted')

    return df.sort_values('adjusted_p').reset_index(drop=True)  # sort by most significant first


# Run for Colorectal: compare colorectal cancer samples vs colorectal healthy controls
print('Running Colorectal Cancer vs Healthy...')
diff_colorectal = run_diff_abundance(abund, meta, 'Colorectal')
sig_col = diff_colorectal[
    (diff_colorectal['adjusted_p'] < 0.05) &            # statistically significant after FDR correction
    (diff_colorectal['log2_fold_change'].abs() > 0.5)   # biologically meaningful: at least 1.4× difference
]
print(f'  Significant genera (Colorectal): {len(sig_col)}')  # 0 genera pass both thresholds

# Run for Breast: compare breast cancer samples vs breast healthy controls
print('Running Breast Cancer vs Healthy...')
diff_breast = run_diff_abundance(abund, meta, 'Breast')
sig_br = diff_breast[
    (diff_breast['adjusted_p'] < 0.05) &
    (diff_breast['log2_fold_change'].abs() > 0.5)
]
print(f'  Significant genera (Breast): {len(sig_br)}')  # 1 genus passes both thresholds

print('\n--- Summary ---')
print(f'Colorectal significant genera: {len(sig_col)}')
print(f'Breast     significant genera: {len(sig_br)}')

In [ ]:
# Cell 4 — Kruskal-Wallis cross-cancer comparison (all 3 cancer types, cancer samples only)
# Goal: find bacteria whose abundance is DIFFERENT across the three types of cancer
# We use only cancer samples here — healthy controls are not part of this comparison
from statsmodels.stats.multitest import multipletests  # BH FDR to control false discoveries

# Select only samples labeled "Cancer" (not "Healthy")
cancer_mask  = meta['condition'] == 'Cancer'
cancer_meta  = meta[cancer_mask]               # metadata for cancer patients only
cancer_abund = abund.loc[cancer_meta.index]    # abundance rows for cancer patients only

# Split cancer abundance by cancer type for the Kruskal-Wallis test
ct_colorectal = cancer_abund.loc[cancer_meta['cancer_type'] == 'Colorectal']  # 123 samples
ct_breast     = cancer_abund.loc[cancer_meta['cancer_type'] == 'Breast']      # 193 samples
ct_prostate   = cancer_abund.loc[cancer_meta['cancer_type'] == 'Prostate']    # 31 samples

print(f'Cancer-only sample counts — Colorectal: {len(ct_colorectal)}, '
      f'Breast: {len(ct_breast)}, Prostate: {len(ct_prostate)}')

kw_results = []
for genus in cancer_abund.columns:  # test each bacterium one at a time
    g1 = ct_colorectal[genus].values  # abundance in colorectal cancer patients
    g2 = ct_breast[genus].values      # abundance in breast cancer patients
    g3 = ct_prostate[genus].values    # abundance in prostate cancer patients

    # Only test genera where we have at least 2 groups with data
    valid_groups = [g for g in [g1, g2, g3] if len(g) > 0]
    if len(valid_groups) < 2:
        continue

    try:
        # Kruskal-Wallis test: non-parametric ANOVA — asks "do any of the 3 groups differ?"
        # It ranks all values from all groups together, then checks if ranks are evenly mixed
        stat, p_val = stats.kruskal(*valid_groups)  # *valid_groups unpacks the list
    except Exception:
        stat, p_val = np.nan, 1.0  # use p=1 if test fails (treated as not significant)

    kw_results.append({
        'genus':           genus,
        'KW_stat':         stat,
        'p_value':         p_val,
        'mean_colorectal': g1.mean() if len(g1) > 0 else np.nan,  # average in colorectal
        'mean_breast':     g2.mean() if len(g2) > 0 else np.nan,  # average in breast
        'mean_prostate':   g3.mean() if len(g3) > 0 else np.nan   # average in prostate
    })

diff_crosscancer = pd.DataFrame(kw_results).dropna(subset=['p_value'])  # remove failed tests

# BH FDR correction across all 259 genera tested simultaneously
reject, adj_p, _, _ = multipletests(diff_crosscancer['p_value'].values, method='fdr_bh')
diff_crosscancer['adjusted_p'] = adj_p  # add corrected p-values to table
diff_crosscancer = diff_crosscancer.sort_values('adjusted_p').reset_index(drop=True)  # sort by significance

sig_cross = diff_crosscancer[diff_crosscancer['adjusted_p'] < 0.05]  # filter to significant genera
print(f'Cross-cancer significant genera (KW, FDR < 0.05): {len(sig_cross)}')  # 250 genera differ
print(diff_crosscancer.head(10)[['genus', 'KW_stat', 'adjusted_p']])  # show top 10 most significant

In [ ]:
# Cell 5 — Pan-cancer enriched genera (shared between Colorectal and Breast)
# "Pan-cancer" means appearing in MORE THAN ONE cancer type — these bacteria may be general cancer markers
# We look for genera enriched (more abundant) in cancer vs healthy in BOTH Colorectal AND Breast

# Get the set of genera that are enriched in colorectal cancer (significant + positive fold change)
enriched_colorectal = set(
    diff_colorectal.loc[
        (diff_colorectal['adjusted_p'] < 0.05) &     # statistically significant
        (diff_colorectal['direction'] == 'enriched'), # more abundant in cancer than healthy
        'genus'
    ]
)

# Get the set of genera that are enriched in breast cancer
enriched_breast = set(
    diff_breast.loc[
        (diff_breast['adjusted_p'] < 0.05) &
        (diff_breast['direction'] == 'enriched'),
        'genus'
    ]
)

# Intersection: genera enriched in BOTH cancer types → potential pan-cancer markers
pancancer_enriched = enriched_colorectal.intersection(enriched_breast)

print(f'Enriched genera in Colorectal: {len(enriched_colorectal)}')
print(f'Enriched genera in Breast    : {len(enriched_breast)}')
print(f'Shared (pan-cancer enriched) : {len(pancancer_enriched)}')  # 0 in this dataset

if pancancer_enriched:
    print('\nPan-cancer enriched genera:')
    for g in sorted(pancancer_enriched):
        print(f'  {g}')
else:
    # 0 shared genera — reflects low statistical power for cancer vs healthy comparisons in this dataset
    print('\nNo pan-cancer enriched genera found (may reflect dataset characteristics).')

pancancer_df = pd.DataFrame(sorted(pancancer_enriched), columns=['genus'])
# Annotate with fold changes and adjusted p-values from both cancer types (for the paper table)
if not pancancer_df.empty:
    fc_col = diff_colorectal.set_index('genus')[['log2_fold_change', 'adjusted_p']].rename(
        columns={'log2_fold_change': 'log2fc_colorectal', 'adjusted_p': 'adjp_colorectal'})
    fc_br  = diff_breast.set_index('genus')[['log2_fold_change', 'adjusted_p']].rename(
        columns={'log2_fold_change': 'log2fc_breast', 'adjusted_p': 'adjp_breast'})
    # Join fold change columns from both datasets onto the shared genus list
    pancancer_df = pancancer_df.set_index('genus').join(fc_col, how='left').join(fc_br, how='left').reset_index()

print('\nPan-cancer enriched genera table:')
print(pancancer_df)

In [ ]:
# Cell 6 — Figure 5: Stacked bar chart of top 20 genera by cancer type (cancer samples only)
# Each stacked bar shows what fraction of the microbiome the top 20 bacteria make up in each cancer type
# Think of it like a pie chart standing upright, one per cancer type

# Keep only cancer patient samples (no healthy controls for this comparison)
cancer_mask  = meta['condition'] == 'Cancer'
cancer_meta  = meta[cancer_mask]
cancer_abund = abund.loc[cancer_meta.index]  # abundance table: cancer patients only

# Compute the average abundance of each genus within each cancer type
# This creates a 3-row table (one row per cancer type) × 259 columns (one per genus)
mean_by_ct = cancer_abund.join(cancer_meta[['cancer_type']]).groupby('cancer_type').mean()

# Find the top 20 genera by their average abundance across ALL cancer types
overall_mean  = mean_by_ct.mean(axis=0)             # one mean per genus, averaged over all 3 cancer types
top20_genera  = overall_mean.nlargest(20).index.tolist()  # names of 20 most abundant genera

plot_data = mean_by_ct[top20_genera].T  # transpose: rows=genera, columns=cancer types (for stacking)

# Create 20 visually distinct colors (one per genus) using the 'tab20' colormap
cmap20  = plt.cm.get_cmap('tab20', 20)
colors  = [cmap20(i) for i in range(20)]

fig, ax = plt.subplots(figsize=(12, 7))

# Build the stacked bars: each genus adds its height on top of the previous genus
cancer_types = plot_data.columns.tolist()  # ['Breast', 'Colorectal', 'Prostate']
bottom = np.zeros(len(cancer_types))       # start stacking from y=0

for i, genus in enumerate(plot_data.index):
    vals   = plot_data.loc[genus, cancer_types].values.astype(float)  # mean abundance per cancer type
    ax.bar(cancer_types, vals, bottom=bottom, color=colors[i], label=genus, width=0.5)
    bottom += vals  # move the starting height up for the next genus

ax.set_xlabel('Cancer Type', fontsize=13)
ax.set_ylabel('Mean Relative Abundance', fontsize=13)
ax.set_title('Top 20 Genera by Cancer Type (Cancer Samples)', fontsize=14, fontweight='bold')
ax.legend(title='Genus', bbox_to_anchor=(1.02, 1), loc='upper left',
          fontsize=8, title_fontsize=9, framealpha=0.8)  # legend outside the chart to avoid overlap
ax.set_ylim(0, 1.05)  # relative abundance goes 0→1, with a small margin
sns.despine(ax=ax)     # remove top and right border lines for a cleaner look
plt.tight_layout()     # adjust spacing so legend doesn't get clipped

fig5_path = os.path.join(FIGURES_DIR, 'fig05_top20_genera_stacked.png')
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')  # save at print quality
plt.close()
print(f'Figure 5 saved: {fig5_path}')

In [ ]:
# Cell 7 — Figure 6: Heatmap of top 50 differentially abundant genera
# A heatmap colors each cell by value — here: log-scale abundance of each bacterium in each group
# Rows = bacteria (genera), Columns = cancer type + condition groups; color = how much of that bacterium is present

# --- Step 1: Choose which genera to show (up to 50 most interesting ones) ---
def top_sig_genera(diff_df, n=25):
    """Return the top n genera sorted by absolute log2 fold change among significant ones."""
    sig = diff_df[diff_df['adjusted_p'] < 0.05].copy()  # only statistically significant genera
    sig['abs_lfc'] = sig['log2_fold_change'].abs()       # absolute fold change (ignores direction)
    return set(sig.nlargest(n, 'abs_lfc')['genus'].tolist())  # return the n genera with biggest differences

top_col = top_sig_genera(diff_colorectal, 25)  # top 25 from colorectal cancer vs healthy
top_br  = top_sig_genera(diff_breast,     25)  # top 25 from breast cancer vs healthy
selected = top_col.union(top_br)               # combine both sets (union can be up to 50)

# If we still have fewer than 50 genera, fill up using cross-cancer significant genera
if len(selected) < 50:
    needed       = 50 - len(selected)
    cross_extras = diff_crosscancer[diff_crosscancer['adjusted_p'] < 0.05]['genus'].tolist()
    for g in cross_extras:
        if g not in selected:
            selected.add(g)
            if len(selected) >= 50:
                break

# If still fewer than 50, pad with the most abundant genera overall
if len(selected) < 50:
    needed      = 50 - len(selected)
    top_overall = overall_mean.nlargest(50 + len(selected)).index.tolist()
    for g in top_overall:
        if g not in selected:
            selected.add(g)
            if len(selected) >= 50:
                break

# Only keep genera that actually exist as columns in the abundance table
selected = [g for g in selected if g in abund.columns]
print(f'Genera selected for heatmap: {len(selected)}')

# --- Step 2: Compute group-mean abundance for each cancer type × condition combination ---
def group_mean(ct, cond):
    """Return mean abundance (across samples) for a given cancer type and condition."""
    idx = meta[(meta['cancer_type'] == ct) & (meta['condition'] == cond)].index
    idx = idx.intersection(abund.index)        # only samples present in both tables
    if len(idx) == 0:
        return pd.Series(np.nan, index=selected)  # return NaN if no samples in this group
    return abund.loc[idx, selected].mean()        # mean across samples for each genus

# Build one column per group: 5 groups total (Prostate has no healthy controls)
matrix = pd.DataFrame({
    'Colorectal_Cancer':  group_mean('Colorectal', 'Cancer'),
    'Colorectal_Healthy': group_mean('Colorectal', 'Healthy'),
    'Breast_Cancer':      group_mean('Breast',     'Cancer'),
    'Breast_Healthy':     group_mean('Breast',     'Healthy'),
    'Prostate_Cancer':    group_mean('Prostate',   'Cancer')
}, index=selected)

matrix = matrix.dropna(axis=1, how='all')   # remove columns where ALL values are NaN
matrix = matrix.dropna(axis=0, how='all')   # remove rows where ALL values are NaN
matrix = matrix.fillna(0)                   # replace remaining NaN with 0 (absent bacteria)

# Log10 transform: converts tiny fractions to a more readable scale
# e.g., 0.001 → -3, 0.01 → -2, 0.1 → -1; the +1e-6 prevents log(0) errors
log_matrix = np.log10(matrix + 1e-6)

# Only cluster if there are multiple rows/columns to cluster (clustering 1 item is meaningless)
row_cluster = log_matrix.shape[0] > 1  # True if more than 1 genus
col_cluster = log_matrix.shape[1] > 1  # True if more than 1 group column

# Clustermap: like a heatmap but also reorders rows and columns by similarity (hierarchical clustering)
# Red = high abundance, Blue = low abundance (RdBu_r colormap)
g = sns.clustermap(
    log_matrix,
    cmap='RdBu_r',           # diverging colormap: red=high, blue=low
    figsize=(14, 16),
    row_cluster=row_cluster,  # rearrange genera so similar bacteria are near each other
    col_cluster=col_cluster,  # rearrange groups so similar groups are near each other
    xticklabels=True,
    yticklabels=True,
    linewidths=0,             # no grid lines between cells (cleaner look)
    cbar_kws={'label': 'log10(mean relative abundance + 1e-6)'}  # color bar label
)
g.ax_heatmap.set_title(
    'Top Differentially Abundant Genera\n(log10 mean relative abundance)',
    fontsize=13, fontweight='bold', pad=12
)
g.ax_heatmap.tick_params(axis='y', labelsize=6)   # small y-axis text (50 genera labels)
g.ax_heatmap.tick_params(axis='x', labelsize=9)   # larger x-axis text (5 group labels)

fig6_path = os.path.join(FIGURES_DIR, 'fig06_heatmap_diffabund.png')
plt.savefig(fig6_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Figure 6 saved: {fig6_path}')

In [ ]:
# Cell 8 — Figure 7: UpSet diagram of shared vs unique taxa
# An UpSet plot is like a Venn diagram for 3+ groups — it shows how many taxa each group uniquely owns
# vs how many taxa are shared between 2 or all 3 groups

# Define which genera are "significant" for each cancer type
sig_colorectal = set(
    diff_colorectal.loc[diff_colorectal['adjusted_p'] < 0.05, 'genus'].tolist()
)  # genera significantly different in colorectal cancer vs healthy

sig_breast = set(
    diff_breast.loc[diff_breast['adjusted_p'] < 0.05, 'genus'].tolist()
)  # genera significantly different in breast cancer vs healthy

# Prostate has no healthy controls → can't do cancer vs healthy comparison
# Instead: use the top 50 most abundant genera in prostate cancer samples as a proxy
prostate_cancer_idx = meta[(meta['cancer_type'] == 'Prostate') & (meta['condition'] == 'Cancer')].index
prostate_cancer_idx = prostate_cancer_idx.intersection(abund.index)  # filter to rows in abundance table
if len(prostate_cancer_idx) > 0:
    prostate_mean = abund.loc[prostate_cancer_idx].mean()  # mean abundance per genus in prostate cancer
    sig_prostate  = set(prostate_mean.nlargest(50).index.tolist())  # top 50 most abundant genera
else:
    sig_prostate = set()  # empty if no samples found

print(f'Significant taxa — Colorectal: {len(sig_colorectal)}, '
      f'Breast: {len(sig_breast)}, Prostate (top50): {len(sig_prostate)}')

# Compute overlap counts for each combination of cancer types
only_col  = sig_colorectal - sig_breast - sig_prostate          # unique to colorectal
only_br   = sig_breast     - sig_colorectal - sig_prostate      # unique to breast
only_pro  = sig_prostate   - sig_colorectal - sig_breast        # unique to prostate
col_br    = (sig_colorectal & sig_breast)   - sig_prostate      # shared by colorectal and breast only
col_pro   = (sig_colorectal & sig_prostate) - sig_breast        # shared by colorectal and prostate only
br_pro    = (sig_breast     & sig_prostate) - sig_colorectal    # shared by breast and prostate only
all_three = sig_colorectal & sig_breast & sig_prostate          # shared by all three cancer types

try:
    # Try to use the upsetplot library for a proper UpSet diagram
    from upsetplot import UpSet, from_memberships

    memberships = []
    all_taxa = sig_colorectal | sig_breast | sig_prostate  # union of all significant taxa
    for taxon in all_taxa:
        # For each taxon, record which cancer types it belongs to
        membership = tuple(sorted([
            ct for ct, s in [
                ('Colorectal', sig_colorectal),
                ('Breast',     sig_breast),
                ('Prostate',   sig_prostate)
            ] if taxon in s  # include only cancer types where this taxon is significant
        ]))
        memberships.append(membership)

    data   = from_memberships(memberships)  # convert list of memberships to UpSet format
    upset  = UpSet(data, subset_size='count', show_counts=True)
    fig    = plt.figure(figsize=(10, 6))
    upset.plot(fig)  # draw the UpSet diagram
    plt.suptitle('UpSet Plot: Shared vs Unique Significant Taxa', fontsize=13, fontweight='bold')
    fig7_path = os.path.join(FIGURES_DIR, 'fig07_upset_shared_taxa.png')
    plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Figure 7 saved (UpSet): {fig7_path}')

except ImportError:
    # upsetplot library not installed — fall back to a simple horizontal bar chart
    print('upsetplot not available. Using fallback horizontal bar chart.')

    labels = [
        'Only Colorectal',
        'Only Breast',
        'Only Prostate',
        'Colorectal & Breast',
        'Colorectal & Prostate',
        'Breast & Prostate',
        'All Three'
    ]
    counts = [
        len(only_col), len(only_br), len(only_pro),
        len(col_br),   len(col_pro), len(br_pro),   len(all_three)
    ]
    colors_upset = [
        '#1f77b4', '#ff7f0e', '#2ca02c',
        '#9467bd', '#8c564b', '#e377c2', '#d62728'
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(labels, counts, color=colors_upset, edgecolor='white')  # horizontal bars
    for bar, count in zip(bars, counts):
        # Print count label at the right end of each bar
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                str(count), va='center', ha='left', fontsize=11)
    ax.set_xlabel('Number of Taxa', fontsize=12)
    ax.set_title('Shared vs Unique Significant Taxa Across Cancer Types', fontsize=13, fontweight='bold')
    ax.set_xlim(0, max(counts) * 1.2 if max(counts) > 0 else 10)  # add extra space for count labels
    sns.despine(ax=ax)
    plt.tight_layout()
    fig7_path = os.path.join(FIGURES_DIR, 'fig07_upset_shared_taxa.png')
    plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Figure 7 saved (fallback bar chart): {fig7_path}')

In [ ]:
# Cell 9 — Figure 8: LDA-style effect size bar chart (top 20 discriminative genera)
# LDA (Linear Discriminant Analysis) effect size measures how strongly a bacterium discriminates between groups
# Here we approximate it with Cohen's d: how many standard deviations apart are two groups' means?
# A large effect size means the bacterium is very different between cancer types — good discriminator

# Get all genera that significantly differ across cancer types (from the Kruskal-Wallis test)
sig_cross_genera = diff_crosscancer[
    diff_crosscancer['adjusted_p'] < 0.05
]['genus'].tolist()

# If fewer than 20 are significant, just take the top 20 by KW statistic
if len(sig_cross_genera) < 20:
    sig_cross_genera = diff_crosscancer.head(20)['genus'].tolist()

def pooled_effect_size(vals_a, vals_b):
    """Cohen's d: how many standard deviations apart are the means of two groups?
    d = |mean_A - mean_B| / pooled_std
    d < 0.2: small, 0.2–0.5: medium, > 0.5: large effect
    """
    mean_diff  = np.abs(np.mean(vals_a) - np.mean(vals_b))           # absolute mean difference
    pooled_std = np.sqrt((np.std(vals_a) ** 2 + np.std(vals_b) ** 2) / 2)  # average of both stds
    return mean_diff / (pooled_std + 1e-10)  # +1e-10 prevents division by zero

lda_results = []

# Group cancer samples by cancer type (we already computed these in Cell 4)
ct_groups = {
    'Colorectal': ct_colorectal,  # 123 colorectal cancer samples
    'Breast':     ct_breast,      # 193 breast cancer samples
    'Prostate':   ct_prostate     # 31 prostate cancer samples
}

for genus in sig_cross_genera:
    if genus not in cancer_abund.columns:
        continue  # skip if genus not in the abundance table

    max_effect = 0          # track the largest pairwise effect size found
    means = {ct: grp[genus].mean() if len(grp) > 0 else 0.0
             for ct, grp in ct_groups.items()}  # mean abundance per cancer type

    # Check all pairwise combinations: Colorectal vs Breast, Colorectal vs Prostate, Breast vs Prostate
    for (ct_a, ct_b) in combinations(list(ct_groups.keys()), 2):
        vals_a = ct_groups[ct_a][genus].values if len(ct_groups[ct_a]) > 0 else np.array([0])
        vals_b = ct_groups[ct_b][genus].values if len(ct_groups[ct_b]) > 0 else np.array([0])
        es     = pooled_effect_size(vals_a, vals_b)  # effect size for this pair
        if es > max_effect:
            max_effect = es  # keep the largest effect size across all pairs

    # The "dominant" cancer type is the one with the highest mean abundance for this genus
    dominant_ct = max(means, key=means.get)
    lda_results.append({
        'genus':               genus,
        'effect_size':         max_effect,
        'dominant_cancer_type': dominant_ct  # which cancer type this bacterium is most associated with
    })

# Keep only top 20 genera sorted by largest effect size
lda_df = pd.DataFrame(lda_results).sort_values('effect_size', ascending=False).head(20)

# Color each bar by which cancer type "owns" that bacterium (highest mean abundance)
ct_palette = {'Colorectal': '#1f77b4', 'Breast': '#ff7f0e', 'Prostate': '#2ca02c'}
bar_colors = [ct_palette.get(ct, '#888888') for ct in lda_df['dominant_cancer_type']]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    lda_df['genus'][::-1],        # reverse order so highest is at the top
    lda_df['effect_size'][::-1],  # matching reversed values
    color=list(reversed(bar_colors)),
    edgecolor='white', linewidth=0.5
)

# Add a legend explaining bar colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in ct_palette.items()]
ax.legend(handles=legend_elements, title='Highest in', loc='lower right', fontsize=9)

ax.set_xlabel('Effect Size (max pairwise Cohen\'s d)', fontsize=12)
ax.set_title('Top 20 Discriminative Genera (LDA-style Effect Size)', fontsize=13, fontweight='bold')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.7, label='d=0.5')  # medium-effect reference line
sns.despine(ax=ax)
plt.tight_layout()

fig8_path = os.path.join(FIGURES_DIR, 'fig08_lda_effect_size.png')
plt.savefig(fig8_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Figure 8 saved: {fig8_path}')

In [ ]:
# Cell 10 — Figure 8b: Volcano plots (Colorectal and Breast)
# A volcano plot shows TWO things at once for every bacterium:
#   X-axis = log2 fold change (how much more/less abundant in cancer vs healthy)
#   Y-axis = -log10(adjusted p-value) (how statistically confident we are)
# Bacteria in the top-right corner: MORE abundant in cancer AND statistically certain = enriched
# Bacteria in the top-left corner: LESS abundant in cancer AND statistically certain = depleted
# Bacteria near the bottom: not significantly different (could be random noise)

def volcano_subplot(ax, diff_df, title, lfc_thresh=0.5, p_thresh=0.05):
    """Draw a single volcano plot on the given axes."""
    df = diff_df.copy()
    # Convert adjusted p-value to -log10 scale: small p (e.g. 0.001) → large -log10 (e.g. 3)
    # clip(lower=1e-30) prevents log10(0) which is undefined
    df['neg_log10_adjp'] = -np.log10(df['adjusted_p'].clip(lower=1e-30))

    # Categorize each genus: significantly enriched, significantly depleted, or not significant
    sig_enrich  = (df['adjusted_p'] < p_thresh) & (df['log2_fold_change'] >  lfc_thresh)  # significant & more in cancer
    sig_deplete = (df['adjusted_p'] < p_thresh) & (df['log2_fold_change'] < -lfc_thresh)  # significant & less in cancer
    ns_mask     = ~(sig_enrich | sig_deplete)  # everything else = not significant

    # Draw dots: grey for not significant, red for enriched, blue for depleted
    ax.scatter(df.loc[ns_mask,    'log2_fold_change'], df.loc[ns_mask,    'neg_log10_adjp'],
               c='#AAAAAA', alpha=0.5, s=18, label='NS')
    ax.scatter(df.loc[sig_enrich, 'log2_fold_change'], df.loc[sig_enrich, 'neg_log10_adjp'],
               c='#d62728', alpha=0.8, s=30, label=f'Enriched (n={sig_enrich.sum()})')
    ax.scatter(df.loc[sig_deplete,'log2_fold_change'], df.loc[sig_deplete,'neg_log10_adjp'],
               c='#1f77b4', alpha=0.8, s=30, label=f'Depleted (n={sig_deplete.sum()})')

    # Dashed reference lines: horizontal = significance threshold, vertical = fold-change threshold
    ax.axhline(-np.log10(p_thresh), color='gray', linestyle='--', linewidth=0.8)   # p=0.05 line
    ax.axvline( lfc_thresh,         color='gray', linestyle='--', linewidth=0.8)   # +0.5 fold change line
    ax.axvline(-lfc_thresh,         color='gray', linestyle='--', linewidth=0.8)   # -0.5 fold change line

    # Label the top 10 most significant genera with their genus name
    top_sig = df[sig_enrich | sig_deplete].nlargest(10, 'neg_log10_adjp')
    label_kwargs = dict(fontsize=6, ha='center', va='bottom')

    try:
        # adjustText library repositions labels to avoid overlapping text
        from adjustText import adjust_text
        texts = []
        for _, row in top_sig.iterrows():
            texts.append(ax.text(row['log2_fold_change'], row['neg_log10_adjp'],
                                 row['genus'], **label_kwargs))
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='black', lw=0.4))
    except ImportError:
        # If adjustText not installed, place labels slightly above each dot without smart repositioning
        for _, row in top_sig.iterrows():
            ax.text(row['log2_fold_change'], row['neg_log10_adjp'] + 0.1,
                    row['genus'], **label_kwargs)

    ax.set_xlabel('log2 Fold Change (Cancer / Healthy)', fontsize=10)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    sns.despine(ax=ax)  # remove top and right borders


fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # two volcano plots side by side

volcano_subplot(axes[0], diff_colorectal, 'Colorectal: Cancer vs Healthy')
volcano_subplot(axes[1], diff_breast,     'Breast: Cancer vs Healthy')

plt.suptitle('Volcano Plots — Differential Abundance (Cancer vs Healthy)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig8b_path = os.path.join(FIGURES_DIR, 'fig08b_volcano_plots.png')
plt.savefig(fig8b_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Figure 8b saved: {fig8b_path}')

In [ ]:
# Cell 11 — Save results
# Save all differential abundance result tables to CSV files for use by later notebooks and the paper

path_col   = os.path.join(RESULTS_DIR, 'diff_abundance_colorectal.csv')   # colorectal cancer vs healthy
path_br    = os.path.join(RESULTS_DIR, 'diff_abundance_breast.csv')        # breast cancer vs healthy
path_cross = os.path.join(RESULTS_DIR, 'diff_abundance_crosscancer.csv')  # cross-cancer Kruskal-Wallis
path_pan   = os.path.join(RESULTS_DIR, 'pancancer_shared_enriched_genera.csv')  # shared enriched genera

diff_colorectal.to_csv(path_col,    index=False)   # save colorectal results (no row numbers)
diff_breast.to_csv(    path_br,     index=False)   # save breast results
diff_crosscancer.to_csv(path_cross, index=False)   # save cross-cancer KW results
pancancer_df.to_csv(   path_pan,    index=False)   # save pan-cancer shared genera (empty here)

print('=== Notebook 05 Complete ===')
print(f'\nResults saved to: {RESULTS_DIR}')
print(f'  {os.path.basename(path_col)}')    # filename only (not full path)
print(f'  {os.path.basename(path_br)}')
print(f'  {os.path.basename(path_cross)}')
print(f'  {os.path.basename(path_pan)}')
print(f'\nFigures saved to: {FIGURES_DIR}')
print(f'  fig05_top20_genera_stacked.png')  # stacked bar chart of top 20 genera
print(f'  fig06_heatmap_diffabund.png')     # heatmap of top 50 differentially abundant genera
print(f'  fig07_upset_shared_taxa.png')     # UpSet/bar chart of shared vs unique taxa
print(f'  fig08_lda_effect_size.png')       # LDA-style effect size bar chart
print(f'  fig08b_volcano_plots.png')        # volcano plots for Colorectal and Breast

print('\n=== Final Summary ===')
print(f'Colorectal diff. genera (FDR<0.05, |LFC|>0.5) : {len(sig_col)}')   # 0 genera
print(f'Breast     diff. genera (FDR<0.05, |LFC|>0.5) : {len(sig_br)}')    # 1 genus
print(f'Cross-cancer KW significant genera (FDR<0.05) : {len(sig_cross)}') # 250 genera
print(f'Pan-cancer enriched genera (shared)           : {len(pancancer_enriched)}')